# Проверка на утечки: искажение будущего

Дополнение к разделу 3.2 [solution.ipynb](solution.ipynb). Тест в основном ноутбуке проверяет только движок признаков. Здесь проверяется весь пайплайн от сырого лога до матрицы признаков.

Идея: если пайплайн не использует будущее, то любые изменения событий с датой `≥ T` не должны менять ничего, что строится для недели `[T, T+7)`, кроме таргета. Поэтому:

1. в сыром логе портим **только** события с датой `≥ T`: случайно меняем длительность и процент просмотра, удаляем 30% таких событий и добавляем 20 тыс. фиктивных;
2. заново собираем всё: тёплых пользователей, кандидатов (популярность и ItemKNN), базовые признаки и признаки грамматики;
3. сравниваем с исходным пайплайном. Совпасть должно всё, кроме таргета, а таргет обязан измениться: это показывает, что порча будущего действительно произошла.

Признаки: все признаки финальных наборов (ручной, найденные жадным поиском со случайным пулом и с пулом от LLM) плюс по одному признаку на каждую комбинацию связи, нормировки и агрегации из пространства грамматики. Проверка идёт для двух дат отсечки: начала валидационной (09.08) и тестовой (16.08) недели.

Сэмпл 20 тыс. пользователей: для проверки утечек размер выборки не важен. Код пайплайна берётся из `solution.ipynb` без изменений, меняется только `N_USERS`.

In [1]:
import gc
import json
from datetime import timedelta
from pathlib import Path

import numpy as np

N_CHECK = 20_000
nb = json.load(open('solution.ipynb', encoding='utf-8'))
code_cells = [''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code']


def pick(first_line=None, contains=None):
    hits = [c for c in code_cells
            if (contains in c if contains else c.strip().split('\n')[0].startswith(first_line))]
    assert len(hits) == 1, (first_line or contains, len(hits))
    return hits[0]


PIPELINE = [
    pick(contains='MIMALLOC_PURGE_DELAY'),
    pick('interactions = ('),
    pick('rng = np.random.default_rng(SEED)'),
    pick('del interactions'),
    pick('N_POP, N_KNN, KNN_NEIGHBORS, KNN_HALF_LIFE'),
    pick('candidates = {}'),
    pick('IDCG = np.cumsum'),
    pick('import duckdb'),
    pick('@dataclass(frozen=True)'),
    pick('NEG_PER_USER = 60'),
    pick("TIMES = ['all'").split('r = np.random.default_rng(SEED)')[0],
    pick('AGE_ORD = ').split('WARM = ')[0],
]
g = {'__name__': '__leak__', 'ON_KAGGLE': Path('/kaggle/input').exists()}
for i, src in enumerate(PIPELINE):
    exec(compile(src.replace('N_USERS = 100_000', f'N_USERS = {N_CHECK}'), f'pipeline_{i}', 'exec'), g)
g['con'].execute('SET enable_progress_bar = false')
print(f"пользователей в сэмпле: {N_CHECK:,}, событий: {g['log'].height:,}")

взаимодействий: 5,476,251
пользователей в логах: 962,179, с анкетой: 840,197


объектов в логах: 15,706, в каталоге: 15,963
период: 2021-03-13 — 2021-08-22


строк в сэмпле: 113,156
кандидаты train: 0.0 с
кандидаты val: 0.0 с
кандидаты test: 0.0 с


WITH e AS (SELECT user_id, item_id, dur, pct, age, pow(0.5, age / 30) AS w, CAST(pct >= 80 AS DOUBLE) AS hit FROM ev_val WHERE TRUE AND pct >= 80), n AS (SELECT user_id, val, SUM(w * hit) AS v FROM e JOIN attr_genre USING (item_id) GROUP BY user_id, val), p AS (SELECT user_id, SUM(w * hit) AS v FROM e JOIN attr_genre USING (item_id) GROUP BY user_id), o AS (SELECT val, SUM(w * hit) AS v FROM e JOIN attr_genre USING (item_id) GROUP BY val), t AS (SELECT SUM(w * hit) AS v FROM e JOIN attr_genre USING (item_id)), x AS (SELECT n.user_id, n.val, (n.v / NULLIF(p.v, 0)) / NULLIF(o.v / NULLIF(t.v, 0), 0) AS f FROM n JOIN p USING (user_id) JOIN o USING (val) CROSS JOIN t), r AS (SELECT c.row_id, MAX(x.f) AS f FROM cand_val c JOIN attr_genre a USING (item_id) JOIN x ON x.user_id = c.user_id AND x.val = a.val GROUP BY c.row_id) SELECT COALESCE(f, 0) AS f FROM (SELECT c.row_id, r.f FROM cand_val c LEFT JOIN r USING (row_id)) ORDER BY row_id
train: событий в истории 88,114, строк кандидатов 88,747,

val: событий в истории 96,193, строк кандидатов 397,524, пользователей 2,253, позитивов 2,506
test: событий в истории 104,483, строк кандидатов 397,396, пользователей 2,244, позитивов 2,619
спецификаций в полном пространстве: 7,389
пользователей в сэмпле: 20,000, событий: 113,156


Признаки для проверки.

In [2]:
Spec, SPACE = g['Spec'], g['SPACE']


def to_spec(name):
    key, *rest = name.split('__')
    for prefix in ('ua_', 'di_'):
        if key.startswith(prefix):
            key = prefix[:2] + ':' + key[3:]
    return Spec(key, *rest)


manual = [s for s in code_cells if s.strip().startswith('MANUAL = [')]
exec(compile(manual[0].split('assert all(')[0], 'manual', 'exec'), g)
final_specs = g['MANUAL'] + [to_spec(n) for f in ('search_greedy_random.json', 'search_greedy_llm.json')
                             for n in json.loads(Path('cache', f).read_text(encoding='utf-8'))['accepted']]
strata = {}
for s in SPACE:                                   # по одному признаку на комбинацию связи, нормировки и агрегации
    strata.setdefault((s.key, s.norm, s.agg), s)
SPECS = list(dict.fromkeys(final_specs + list(strata.values())))
print(f'признаков финальных наборов: {len(set(final_specs))}, всего в проверке: {len(SPECS)}')

признаков финальных наборов: 41, всего в проверке: 163


Искажение будущего: меняются и удаляются только события с датой `≥ T`, фиктивные события получают даты от `T` до конца данных. Пары, которые уже есть в прошлом, фиктивными не перезаписываются: в KION одна строка на пару.

In [3]:
import polars as pl


def corrupt_future(log, start, seed=0):
    r = np.random.default_rng(seed)
    past = log.filter(pl.col('dt') < start)
    fut = log.filter(pl.col('dt') >= start)
    fut = fut.with_columns(
        total_dur=(pl.col('total_dur') * pl.Series(r.uniform(0, 5, fut.height))).cast(pl.Int64),
        watched_pct=pl.Series(r.uniform(0, 100, fut.height)),
    ).sample(fraction=0.7, seed=seed)
    n_fake = 20_000
    last = log['dt'].max()
    fake = pl.DataFrame({
        'user_id': r.choice(log['user_id'].unique().to_numpy(), n_fake),
        'item_id': r.choice(log['item_id'].unique().to_numpy(), n_fake),
        'dt': [start + timedelta(days=int(d)) for d in r.integers(0, (last - start).days + 1, n_fake)],
        'total_dur': r.integers(1, 20_000, n_fake),
        'watched_pct': r.uniform(0, 100, n_fake),
    }).select(log.columns).cast(log.schema)
    out = pl.concat([past, fut, fake]).unique(['user_id', 'item_id'], keep='first', maintain_order=True)
    assert out.filter(pl.col('dt') < start).sort(['user_id', 'item_id']).equals(past.sort(['user_id', 'item_id']))
    return out


def build(source_log, start, tag, keep_users=None):
    """Весь пайплайн для недели [start, start+7): кандидаты, базовые признаки, признаки грамматики."""
    c = g['build_candidates'](source_log, start)
    if keep_users is not None:
        c = c.filter(pl.col('user_id').is_in(pl.Series(keep_users).implode()))
    g['setup_fold'](tag, start, source_log=source_log)
    g['cand'][tag] = g['make_cand_table'](tag, c)
    xb = g['base_matrix'](tag)
    xf = g['compute_features'](SPECS, tag, cache=False)
    cc = g['cand'][tag]
    g['con'].execute(f'DROP TABLE ev_{tag}; DROP TABLE cand_{tag}')
    del g['cand'][tag]
    gc.collect()
    return cc, xb, xf


def same(a, b):
    return a.shape == b.shape and np.allclose(a, b, rtol=1e-5, atol=1e-6, equal_nan=True)


BASE = g['BASE']
POP, KNN = BASE.index('pop_score'), BASE.index('knn_score')
POP_R, KNN_R = BASE.index('pop_rank'), BASE.index('knn_rank')
STATIC = [i for i, c in enumerate(BASE) if c.startswith(('i_', 'u_'))]


def compare(a, b):
    """Совмещает строки по паре (пользователь, тайтл) и сравнивает всё, что для них посчитано."""
    ka = a[0].select('user_id', 'item_id').with_row_index('ia')
    kb = b[0].select('user_id', 'item_id').with_row_index('ib')
    m = ka.join(kb, on=['user_id', 'item_id'], how='inner')
    xa, xb = a[1][m['ia'].to_numpy()], b[1][m['ib'].to_numpy()]
    in_pop = (xa[:, POP_R] < 999) & (xb[:, POP_R] < 999)      # пара пришла из списка популярного в обоих
    in_knn = (xa[:, KNN_R] < 999) & (xb[:, KNN_R] < 999)
    return {
        'совпало пар кандидатов': f'{len(m) / max(len(ka), len(kb)):.2%}',
        'скор популярности': same(xa[in_pop, POP], xb[in_pop, POP]),
        'скор ItemKNN': same(xa[in_knn, KNN], xb[in_knn, KNN]),
        'анкета и атрибуты тайтла': same(xa[:, STATIC], xb[:, STATIC]),
        'ранги генераторов': same(xa[:, [POP_R, KNN_R]], xb[:, [POP_R, KNN_R]]),
        f'{len(SPECS)} признаков грамматики': same(a[2][m['ia'].to_numpy()], b[2][m['ib'].to_numpy()]),
    }

Сравнение. Ранжируются только пользователи, активные в целевой неделе (протокол оценки, раздел 2.2): этот набор по определению зависит от будущего, и искажение его меняет — фиктивные события делают активными новых пользователей, удаление части событий убирает старых. Поэтому сравниваем всё, что строится **для одних и тех же пользователей**: тех, кто активен и в исходном, и в искажённом логе.

Исходный пайплайн собирается дважды, и сначала сравниваются два одинаковых запуска. Генератор популярности не фиксирует порядок тайтлов с равной популярностью (README, раздел «Два независимых прогона»), поэтому даже одинаковые запуски дают немного разные наборы кандидатов. Строки совмещаются по паре (пользователь, тайтл): признаки пары не зависят от того, какие ещё пары попали в кандидаты. Если между исходным и искажённым логом расхождения такие же, как между двумя одинаковыми запусками, их причина — невоспроизводимость, а не утечка.

In [4]:
rows = []
for name, start in [('val', g['FOLDS']['val']), ('test', g['FOLDS']['test'])]:
    log = g['log']
    bad = corrupt_future(log, start)
    warm_ref = g['split'](log, start)[1]['user_id'].unique()
    warm_bad = g['split'](bad, start)[1]['user_id'].unique()
    common = np.intersect1d(warm_ref.to_numpy(), warm_bad.to_numpy())
    ref1 = build(log, start, f'{name}_ref1', common)
    ref2 = build(log, start, f'{name}_ref2', common)
    cor = build(bad, start, f'{name}_bad', common)
    for label, other in [('повтор исходного', ref2), ('искажённое будущее', cor)]:
        rows.append({
            'отсечка': str(start), 'сравнение с исходным': label,
            'общих активных пользователей': len(common),
            **compare(ref1, other),
            'позитивов в таргете': f"{ref1[0]['target'].sum():,} → {other[0].filter(pl.col('user_id').is_in(pl.Series(common).implode()))['target'].sum():,}",
        })
    del ref1, ref2, cor
    gc.collect()
result = pl.DataFrame(rows)
result

отсечка,сравнение с исходным,общих активных пользователей,совпало пар кандидатов,скор популярности,скор ItemKNN,анкета и атрибуты тайтла,ранги генераторов,163 признаков грамматики,позитивов в таргете
str,str,i64,str,bool,bool,bool,bool,bool,str
"""2021-08-09""","""повтор исходного""",2011,"""99.95%""",true,true,true,false,true,"""2,345 → 2,344"""
"""2021-08-09""","""искажённое будущее""",2011,"""99.96%""",true,true,true,false,true,"""2,345 → 1,736"""
"""2021-08-16""","""повтор исходного""",2115,"""99.79%""",true,true,true,false,true,"""2,530 → 2,529"""
"""2021-08-16""","""искажённое будущее""",2115,"""99.92%""",true,true,true,false,true,"""2,530 → 1,853"""


**Итог.** Для обеих дат отсечки всё, что модель знает о паре пользователь–тайтл, от будущего не зависит: скор популярности, скор ItemKNN, анкета, атрибуты тайтла и все 163 признака грамматики совпадают с исходными, хотя таргет после искажения заметно изменился (2 345 → 1 736 и 2 530 → 1 853 позитивов). Утечки нет.

Различаются две вещи, и обе различаются точно так же между двумя одинаковыми запусками на неизменённом логе. Значит, их причина — невоспроизводимость, а не будущее:

- состав кандидатов совпадает на 99.8–99.96%: тайтлы с равной популярностью на границе топ-100 попадают в список то одни, то другие. Это задевает и позитивы: в повторном запуске один позитив выпал из кандидатов;
- ранги генераторов (`pop_rank`, `knn_rank`) при равных скорах переставляются.

Исправление — сортировка с явным вторым ключом (`item_id`) в `popular_candidates` и при расчёте рангов. В основном ноутбуке оно не применено, чтобы код совпадал с опубликованными результатами.

Набор ранжируемых пользователей по протоколу оценки определяется активностью в целевой неделе, поэтому при искажении он меняется (здесь сравнивались только общие пользователи). На признаки конкретного пользователя это не влияет.